In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
import numpy as np
import os

# Define hyperparameters
embedding_dim = 200
hidden_units = 256
max_sequence_length = 50
image_feature_size = 2048
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')

# Download and preprocess dataset (e.g., MS COCO dataset)
# Load images and captions

# Preprocess the text data
tokenizer.fit_on_texts(captions)
sequences = tokenizer.texts_to_sequences(captions)
max_sequence_length = max(len(seq) for seq in sequences)
vocab_size = len(tokenizer.word_index) + 1

# Pad sequences
captions = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

# Load pre-trained InceptionV3 model for feature extraction
image_model = InceptionV3(include_top=False, weights='imagenet')
new_input = image_model.input
hidden_layer = image_model.layers[-1].output
image_features_extract_model = Model(inputs=new_input, outputs=hidden_layer)

# Extract image features and prepare them for input
image_features = image_features_extract_model.predict(images)
image_features = tf.image.resize(image_features, (8, 8))  # Adjust to match the text input size

# Define the model architecture
# Encoder (for images)
image_input = layers.Input(shape=(8, 8, 2048))
image_flatten = layers.Flatten()(image_input)
image_dense = layers.Dense(hidden_units, activation='relu')(image_flatten)

# Decoder (for captions)
caption_input = layers.Input(shape=(max_sequence_length,))
caption_embedding = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)(caption_input)
caption_lstm = layers.LSTM(hidden_units)(caption_embedding)

# Combine image and caption processing
merged = layers.concatenate([image_dense, caption_lstm])
output = layers.Dense(vocab_size, activation='softmax')(merged)

model = Model(inputs=[image_input, caption_input], outputs=output)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train the model using the prepared data
model.fit([image_features, captions], target_captions, epochs=10, batch_size=64)

# Generate captions for new images
# Load and preprocess new images
new_images = preprocess_images(new_images)
new_image_features = image_features_extract_model.predict(new_images)

# Generate captions using the trained model
predicted_captions = generate_captions(model, new_image_features, tokenizer)

# Print or save the generated captions

# Helper function for generating captions
def generate_captions(model, image_features, tokenizer):
    start_sequence = tokenizer.texts_to_sequences(["<start>"])[0]
    result = []

    for i in range(max_sequence_length):
        input_sequence = np.array([start_sequence])
        next_word_probs = model.predict([image_features, input_sequence])[0]
        next_word_idx = np.argmax(next_word_probs)
        next_word = tokenizer.index_word[next_word_idx]

        if next_word == "<end>":
            break

        result.append(next_word)
        start_sequence = [next_word_idx]

    return " ".join(result)

# Save and load the trained model for later use
model.save('image_caption_model.h5')
loaded_model = keras.models.load_model('image_caption_model.h5')


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load your dataset and preprocess it. You need image and caption data.

# Dummy data for illustration
image_data = ['path_to_image_1.jpg', 'path_to_image_2.jpg']
captions = ['a cat is sitting on a table', 'a dog is running in the park']

# Define hyperparameters
embedding_dim = 256
hidden_units = 512
max_sequence_length = 20
vocab_size = 10000

# Create a tokenizer for captions
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(captions)
vocab_size = len(tokenizer.word_index) + 1

# Preprocess the text data
sequences = tokenizer.texts_to_sequences(captions)
max_sequence_length = max(len(seq) for seq in sequences)
captions = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

# Load the pre-trained InceptionV3 model for feature extraction
image_model = InceptionV3(include_top=False, weights='imagenet')
new_input = image_model.input
hidden_layer = image_model.layers[-1].output
image_features_extract_model = Model(inputs=new_input, outputs=hidden_layer)

# Extract image features and prepare them for input
def preprocess_image(image_path):
    img = load_img(image_path, target_size=(299, 299))
    img = img_to_array(img)
    img = preprocess_input(img)
    return img

image_features = []
for image_path in image_data:
    img = preprocess_image(image_path)
    img = image_features_extract_model.predict(np.expand_dims(img, axis=0))
    img = tf.image.resize(img, (8, 8))
    image_features.append(img)

image_features = np.array(image_features)
image_features = image_features.reshape(image_features.shape[0], -1)

# Define the model architecture
image_input = keras.layers.Input(shape=(64 * 64 * 3))
image_dense = keras.layers.Dense(hidden_units, activation='relu')(image_input)

caption_input = keras.layers.Input(shape=(max_sequence_length,))
caption_embedding = Embedding(input_dim=vocab_size, output_dim=embedding_dim)(caption_input)
caption_lstm = LSTM(hidden_units)(caption_embedding)

decoder_input = keras.layers.concatenate([image_dense, caption_lstm])
output = Dense(vocab_size, activation='softmax')(decoder_input)

model = Model(inputs=[image_input, caption_input], outputs=output)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train the model using the prepared data
# You need to prepare your training data and labels and fit the model here.

# Inference (Generating captions for new images)
# Load the trained model weights
# Preprocess new images
# Generate captions for new images

# Save and load the model for future use
model.save('image_caption_model.h5')
loaded_model = keras.models.load_model('image_caption_model.h5')
